In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

In [0]:
# ── Step 1: Read Bronze, Cast Types, Drop Metadata Columns ──
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def typed(name, dtype="string"):
    """Safe cast: blank -> NULL, unparseable -> NULL (never throws). Returns a column named `name`."""
    if dtype == "string":
        return F.col(name).cast("string").alias(name)
    return F.expr(f"try_cast(nullif(trim(cast(`{name}` as string)), '') as {dtype})").alias(name)

bronze_df = spark.table(f"{CATALOG}.bronze.lineitem")

# Metadata columns (_ingestion_timestamp, _source_file, _pipeline_run_id) are NOT selected, so they are dropped.
# _row_seq remembers the source order so "keep the first occurrence" is deterministic on every rerun.
typed_df = (bronze_df.select(
    typed("l_orderkey", "int"),
    typed("l_partkey", "int"),
    typed("l_suppkey", "int"),
    typed("l_linenumber", "int"),
    typed("l_quantity", "decimal(18,2)"),
    typed("l_extendedprice", "decimal(18,2)"),
    typed("l_discount", "decimal(18,2)"),
    typed("l_tax", "decimal(18,2)"),
    typed("l_returnflag", "string"),
    typed("l_linestatus", "string"),
    typed("l_shipdate", "date"),
    typed("l_commitdate", "date"),
    typed("l_receiptdate", "date"),
    typed("l_shipinstruct", "string"),
    typed("l_shipmode", "string"),
    typed("l_comment", "string")
    )
    .withColumn("_row_seq", F.monotonically_increasing_id()))

bronze_count = bronze_df.count()
print(f"Bronze lineitem rows read: {bronze_count}")
display(typed_df.drop("_row_seq").limit(10))

In [0]:
# ── Step 2: Primary Key Validation (COMPOSITE KEY) ──
# Rule: the row is identified by the combination (l_orderkey, l_linenumber).
# The SAME two conditions apply as for single-column keys:
#   1. Neither key component may be NULL -> NULL_PK
#   2. The combination must be unique       -> DUPLICATE_PK (first occurrence kept, later repeats quarantined)

pk_null = F.col("l_orderkey").isNull() | F.col("l_linenumber").isNull()
pk_window = Window.partitionBy("l_orderkey", "l_linenumber").orderBy("_row_seq")      # NO "desc" and NO running count: plain row_number

flagged = (typed_df
    .withColumn("is_null_pk", pk_null)
    .withColumn("_pk_rn", F.when(pk_null, F.lit(0)).otherwise(F.row_number().over(pk_window)))
    .withColumn("is_duplicate_pk", F.col("_pk_rn") > 1))

print(f"NULL_PK failures:      {flagged.filter('is_null_pk').count()}")
print(f"DUPLICATE_PK failures: {flagged.filter('is_duplicate_pk').count()}")

In [0]:
# ── Step 3: Numeric Range Validation ──
# Line items are real commercial transactions:
#   Rule: l_linenumber      — line numbers start at 1, so must be >= 1.
#   Rule: l_quantity        — quantity ordered must be strictly positive (> 0).
#   Rule: l_extendedprice   — monetary value must be strictly positive (> 0).
#   Rule: l_discount        — proportional reduction, must be within 0% .. 10% (0.00 .. 0.10).
#   Rule: l_tax             — applied on the discounted price, must be within 0% .. 8% (0.00 .. 0.08).
# Zero, negative, or unparseable (NULL after casting) values are invalid.
# Failure codes: INVALID_LINENUMBER, INVALID_QUANTITY, INVALID_PRICE, INVALID_DISCOUNT, INVALID_TAX

DISCOUNT_MAX = 0.10   # 10%
TAX_MAX      = 0.08   # 8%

flagged = (flagged
    .withColumn("is_invalid_linenumber", F.col("l_linenumber").isNull() | (F.col("l_linenumber") < 1))
    .withColumn("is_invalid_quantity",   F.col("l_quantity").isNull()  | (F.col("l_quantity")  <= 0))
    .withColumn("is_invalid_price",      F.col("l_extendedprice").isNull() | (F.col("l_extendedprice") <= 0))
    .withColumn("is_invalid_discount",
        F.col("l_discount").isNull() | (F.col("l_discount") < 0) | (F.col("l_discount") > DISCOUNT_MAX))
    .withColumn("is_invalid_tax",
        F.col("l_tax").isNull() | (F.col("l_tax") < 0) | (F.col("l_tax") > TAX_MAX)))

print(f"INVALID_LINENUMBER failures: {flagged.filter('is_invalid_linenumber').count()}")
print(f"INVALID_QUANTITY failures:   {flagged.filter('is_invalid_quantity').count()}")
print(f"INVALID_PRICE failures:      {flagged.filter('is_invalid_price').count()}")
print(f"INVALID_DISCOUNT failures:   {flagged.filter('is_invalid_discount').count()}")
print(f"INVALID_TAX failures:        {flagged.filter('is_invalid_tax').count()}")

In [0]:
# ── Step 4: Date Range Validation ──
# Rule: the dataset covers a defined historical operating period running
#       from 1 January 1992 to 31 December 1998.
# The lineitem table captures several shipping date milestones — EVERY date field
# (l_shipdate, l_commitdate, l_receiptdate) must fall inside this window.
# A NULL or out-of-window date is invalid.
# Failure codes: INVALID_SHIPDATE, INVALID_COMMITDATE, INVALID_RECEIPTDATE

DATE_FROM = "1992-01-01"
DATE_TO   = "1998-12-31"

def out_of_window(col):
    """True when the date is NULL or outside the operating window."""
    return (F.col(col).isNull()
            | (F.col(col) < F.lit(DATE_FROM))
            | (F.col(col) > F.lit(DATE_TO)))

flagged = (flagged
    .withColumn("is_invalid_shipdate",    out_of_window("l_shipdate"))
    .withColumn("is_invalid_commitdate", out_of_window("l_commitdate"))
    .withColumn("is_invalid_receiptdate", out_of_window("l_receiptdate")))

print(f"INVALID_SHIPDATE failures:     {flagged.filter('is_invalid_shipdate').count()}")
print(f"INVALID_COMMITDATE failures:   {flagged.filter('is_invalid_commitdate').count()}")
print(f"INVALID_RECEIPTDATE failures:  {flagged.filter('is_invalid_receiptdate').count()}")

In [0]:
# ── Step 5: Referential Integrity Validation ──
# Each line item belongs to an order, references a product, and was fulfilled by a supplier:
#   l_orderkey -> silver.orders    (the order the line belongs to)
#   l_partkey  -> silver.part      (the product ordered)
#   l_suppkey  -> silver.supplier  (the supplier who fulfilled it)
# All three parents are already cleaned in Silver (parent-before-child order).
# A NULL or unmatched key is unresolvable -> quarantined.

fk_df = flagged

orders_parent = (spark.table(f"{CATALOG}.silver.orders")
    .select(F.col("o_orderkey").alias("_order_pk")).distinct()
    .withColumn("_order_hit", F.lit(1)))
fk_df = (fk_df.join(orders_parent, F.col("l_orderkey") == F.col("_order_pk"), "left")
    .withColumn("is_order", F.col("_order_hit").isNull())
    .drop("_order_pk", "_order_hit"))

part_parent = (spark.table(f"{CATALOG}.silver.part")
    .select(F.col("p_partkey").alias("_part_pk")).distinct()
    .withColumn("_part_hit", F.lit(1)))
fk_df = (fk_df.join(part_parent, F.col("l_partkey") == F.col("_part_pk"), "left")
    .withColumn("is_part", F.col("_part_hit").isNull())
    .drop("_part_pk", "_part_hit"))

supplier_parent = (spark.table(f"{CATALOG}.silver.supplier")
    .select(F.col("s_suppkey").alias("_supplier_pk")).distinct()
    .withColumn("_supplier_hit", F.lit(1)))
fk_df = (fk_df.join(supplier_parent, F.col("l_suppkey") == F.col("_supplier_pk"), "left")
    .withColumn("is_supplier", F.col("_supplier_hit").isNull())
    .drop("_supplier_pk", "_supplier_hit"))

flagged = fk_df

print(f"INVALID_FK_ORDER failures:    {flagged.filter('is_order').count()}")
print(f"INVALID_FK_PART failures:     {flagged.filter('is_part').count()}")
print(f"INVALID_FK_SUPPLIER failures: {flagged.filter('is_supplier').count()}")

In [0]:
# ── Step 6: Build Rejection Reasons, Split Clean vs Quarantine, Write Tables ──
flagged = flagged.withColumn(
    "rejection_reason",
    F.concat_ws(", ",
        F.when(F.col("is_null_pk"), F.lit("NULL_PK")),
        F.when(F.col("is_duplicate_pk"), F.lit("DUPLICATE_PK")),
        F.when(F.col("is_invalid_linenumber"), F.lit("INVALID_LINENUMBER")),
        F.when(F.col("is_invalid_quantity"), F.lit("INVALID_QUANTITY")),
        F.when(F.col("is_invalid_price"), F.lit("INVALID_PRICE")),
        F.when(F.col("is_invalid_discount"), F.lit("INVALID_DISCOUNT")),
        F.when(F.col("is_invalid_tax"), F.lit("INVALID_TAX")),
        F.when(F.col("is_invalid_shipdate"), F.lit("INVALID_SHIPDATE")),
        F.when(F.col("is_invalid_commitdate"), F.lit("INVALID_COMMITDATE")),
        F.when(F.col("is_invalid_receiptdate"), F.lit("INVALID_RECEIPTDATE")),
        F.when(F.col("is_order"), F.lit("INVALID_FK_ORDER")),
        F.when(F.col("is_part"), F.lit("INVALID_FK_PART")),
        F.when(F.col("is_supplier"), F.lit("INVALID_FK_SUPPLIER"))
    ),
)

out_cols = ["l_orderkey", "l_partkey", "l_suppkey", "l_linenumber", "l_quantity", "l_extendedprice",
            "l_discount", "l_tax", "l_returnflag", "l_linestatus", "l_shipdate", "l_commitdate",
            "l_receiptdate", "l_shipinstruct", "l_shipmode", "l_comment"]

silver_df     = flagged.filter(F.col("rejection_reason") == "").select(*out_cols)
quarantine_df = flagged.filter(F.col("rejection_reason") != "").select(*out_cols, "rejection_reason")

silver_count = silver_df.count()
quarantine_count = quarantine_df.count()
print(f"Silver rows:     {silver_count}")
print(f"Quarantine rows: {quarantine_count}")

# overwrite => rerunning gives the same result
(silver_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.lineitem"))
(quarantine_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.lineitem_quarantine"))

print("Tables written: silver.lineitem, silver.lineitem_quarantine")

In [0]:
# ── Step 7: Bronze-Silver-Quarantine Reconciliation ──
bronze_total     = spark.table(f"{CATALOG}.bronze.lineitem").count()
silver_total     = spark.table(f"{CATALOG}.silver.lineitem").count()
quarantine_total = spark.table(f"{CATALOG}.silver.lineitem_quarantine").count()

print(f"Bronze row count:      {bronze_total}")
print(f"Silver row count:      {silver_total}")
print(f"Quarantine row count:  {quarantine_total}")
print(f"Silver + Quarantine:   {silver_total + quarantine_total}")

assert bronze_total == silver_total + quarantine_total, (
    f"RECONCILIATION FAILED: Bronze ({bronze_total}) != Silver ({silver_total}) + Quarantine ({quarantine_total})"
)
print("Reconciliation PASSED: Bronze = Silver + Quarantine")

In [0]:
# ── Step 8: Data Quality Report ──
# Every check is listed, even when zero rows fail.
rules = [
    ("NULL_PK", "(l_orderkey, l_linenumber) — neither key component may be null", "is_null_pk"),
    ("DUPLICATE_PK", "(l_orderkey, l_linenumber) combination must be unique (first occurrence kept)", "is_duplicate_pk"),
    ("INVALID_LINENUMBER", "l_linenumber must be >= 1", "is_invalid_linenumber"),
    ("INVALID_QUANTITY", "l_quantity must be strictly positive (> 0)", "is_invalid_quantity"),
    ("INVALID_PRICE", "l_extendedprice must be strictly positive (> 0)", "is_invalid_price"),
    ("INVALID_DISCOUNT", "l_discount must be within 0.00 .. 0.10 (0% to 10%)", "is_invalid_discount"),
    ("INVALID_TAX", "l_tax must be within 0.00 .. 0.08 (0% to 8%)", "is_invalid_tax"),
    ("INVALID_SHIPDATE", f"l_shipdate must be within {DATE_FROM} .. {DATE_TO}", "is_invalid_shipdate"),
    ("INVALID_COMMITDATE", f"l_commitdate must be within {DATE_FROM} .. {DATE_TO}", "is_invalid_commitdate"),
    ("INVALID_RECEIPTDATE", f"l_receiptdate must be within {DATE_FROM} .. {DATE_TO}", "is_invalid_receiptdate"),
    ("INVALID_FK_ORDER", "l_orderkey must exist in silver.orders", "is_order"),
    ("INVALID_FK_PART", "l_partkey must exist in silver.part", "is_part"),
    ("INVALID_FK_SUPPLIER", "l_suppkey must exist in silver.supplier", "is_supplier")
]
dq_rows = [("lineitem", code, desc, flagged.filter(F.col(flag)).count(), "Quarantine") for code, desc, flag in rules]

dq_report = spark.createDataFrame(
    dq_rows, ["table", "rule_code", "check_description", "failing_rows", "action_taken"]
)
display(dq_report)

# Persist so the whole run is auditable (overwrite per table is not needed: one table per notebook)
(dq_report.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.dq_report_lineitem"))

if quarantine_count > 0:
    print(f"\n── Quarantined Records ({quarantine_count} rows) ──")
    display(spark.table(f"{CATALOG}.silver.lineitem_quarantine").limit(10))
else:
    print("\nNo quarantined records — all rows passed every check.")

print(f"\n── Silver lineitem ({silver_count} rows) ──")
display(spark.table(f"{CATALOG}.silver.lineitem").limit(20))